# StudyMate — Transformer Comparison

This notebook evaluates a **pre-trained SQuAD 2.0 transformer** on our
answerability task, to compare it against our other two methods:

| Method | F1 |
|---|---|
| Keyword baseline | ~0.50 |
| Embedding model (mpnet) | 0.644 |
| **Transformer (this notebook)** | ? |

**Note on the task:** the transformer does span-extraction QA — it returns an
answer, or abstains when the question is unanswerable. We convert that to our
yes/no answerability label: *abstains → unanswerable (0)*, *returns a span →
answerable (1)*. This is a fair adaptation of a span-QA model to our task.

**Where to run:** Local Jupyter works (CPU, slower). For speed, upload this
notebook to Google Colab and enable GPU (Runtime → Change runtime type → GPU).


## Step 1 — Install libraries
(Run once. `torch` is a large download.)

In [1]:
!pip install transformers torch

In [2]:
!pip install -U "transformers>=4.50" "tokenizers"

  Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl.metadata (10 kB)


## Step 2 — Imports and configuration

In [2]:
import json, random, time

DATA_PATH = "dev-v2.0.json"          # must be in the same folder as this notebook
MODEL_NAME = "deepset/tinyroberta-squad2"   # small & fast; see note below
N_SAMPLES = 800                      # match our embedding test-set size
SEED = 42

# For a stronger (slower) model, swap MODEL_NAME to:
#   "deepset/roberta-base-squad2"

## Step 3 — Load a balanced sample from SQuAD 2.0
Equal answerable / unanswerable, so accuracy is meaningful.

In [3]:
def load_balanced(path, n, seed=SEED):
    data = json.load(open(path, encoding="utf-8"))
    ans, unans = [], []
    for article in data["data"]:
        for para in article["paragraphs"]:
            ctx = para["context"]
            for qa in para["qas"]:
                item = {"question": qa["question"], "context": ctx,
                        "answerable": 0 if qa["is_impossible"] else 1}
                (unans if qa["is_impossible"] else ans).append(item)
    random.seed(seed)
    random.shuffle(ans); random.shuffle(unans)
    half = n // 2
    rows = ans[:half] + unans[:half]
    random.shuffle(rows)
    return rows

rows = load_balanced(DATA_PATH, N_SAMPLES)
print(f"Loaded {len(rows)} examples "
      f"({sum(r['answerable'] for r in rows)} answerable, "
      f"{sum(1-r['answerable'] for r in rows)} unanswerable)")

Loaded 800 examples (400 answerable, 400 unanswerable)


## Step 4 — Load the transformer
`handle_impossible_answer=True` is the key setting — it lets the model abstain.

In [4]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
print(f"Loading {MODEL_NAME} (first run downloads it)...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model ready.")

Device: cpu
Loading deepset/tinyroberta-squad2 (first run downloads it)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model ready.


## Step 5 — Evaluate
This is the slow part on CPU. Progress prints every 100 questions.

In [5]:
import torch

def is_answerable(question, context):
    """Run the QA model directly. If the best answer span scores no higher
    than the 'no-answer' option (the [CLS] position), we call it UNanswerable."""
    inputs = tokenizer(question, context, return_tensors="pt",
                       truncation=True, max_length=384).to(device)
    with torch.no_grad():
        out = model(**inputs)
    start = out.start_logits[0]
    end = out.end_logits[0]
    # score of the best real answer span vs the "no answer" score (index 0 = [CLS])
    best_span_score = (start.max() + end.max()).item()
    no_answer_score = (start[0] + end[0]).item()
    return 1 if best_span_score > no_answer_score else 0

tp = tn = fp = fn = correct = 0
import time; t0 = time.time()

for i, r in enumerate(rows):
    predicted = is_answerable(r["question"], r["context"])
    actual = r["answerable"]
    if predicted == actual: correct += 1
    if actual == 1 and predicted == 1: tp += 1
    elif actual == 0 and predicted == 0: tn += 1
    elif actual == 0 and predicted == 1: fp += 1
    elif actual == 1 and predicted == 0: fn += 1
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(rows)} done")

print(f"Finished in {time.time()-t0:.0f}s")

  100/800 done
  200/800 done
  300/800 done
  400/800 done
  500/800 done
  600/800 done
  700/800 done
  800/800 done
Finished in 198s


## Step 6 — Results

In [6]:
acc = correct / len(rows)
prec = tp / (tp + fp) if (tp + fp) else 0
rec = tp / (tp + fn) if (tp + fn) else 0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0

print("=== TRANSFORMER RESULTS (answerability task) ===")
print(f"Model    : {MODEL_NAME}")
print(f"Accuracy : {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall   : {rec:.3f}")
print(f"F1 score : {f1:.3f}")
print()
print("Confusion matrix:")
print("                 predicted_unans  predicted_ans")
print(f"  actual_unans        {tn:5d}          {fp:5d}")
print(f"  actual_ans          {fn:5d}          {tp:5d}")
print()
print("=== THREE-WAY COMPARISON ===")
print("  keyword baseline  ~0.50")
print("  embedding model   F1 0.644")
print(f"  transformer       F1 {f1:.3f}")

=== TRANSFORMER RESULTS (answerability task) ===
Model    : deepset/tinyroberta-squad2
Accuracy : 0.851
Precision: 0.819
Recall   : 0.902
F1 score : 0.859

Confusion matrix:
                 predicted_unans  predicted_ans
  actual_unans          320             80
  actual_ans             39            361

=== THREE-WAY COMPARISON ===
  keyword baseline  ~0.50
  embedding model   F1 0.644
  transformer       F1 0.859


## What to do with the result

Whatever F1 you get, you now have a **three-way method comparison** — the core
of a strong results section:

- If the transformer clearly wins → shows the accuracy ceiling of advanced
  methods, and frames your interpretable embedding model as the
  lightweight/explainable trade-off.
- If it's close → an interesting finding that simpler methods compete well.

Either way, record the number for your report. Next step: test all three
methods on a small set of **real student notes** (the novel contribution).
